# Digital Image Processing Lab - Assignment 8
**Problem Statement:**
Write a program to generate following noises using equations derived from PDFs of noise distributions. Compare your output with those generated using inbuilt function `imnoise` / library functions. Plot the histograms of generated noise to determine the shape of the distribution:
1. **Uniform noise**
2. **Gaussian noise**
3. **Erlang noise**
4. **Exponential noise**
5. **Rayleigh noise**

---
### Mathematical Derivations of Noise Distributions
Let $U, U_1, U_2 \sim \text{Uniform}(0, 1)$.

1. **Uniform Noise:**
   $$p(z) = \begin{cases} \frac{1}{b - a} & a \le z \le b \\ 0 & \text{otherwise} \end{cases} \implies z = a + (b - a) U$$

2. **Gaussian Noise:**
   $$p(z) = \frac{1}{\sqrt{2\pi}\sigma} e^{-\frac{(z-\mu)^2}{2\sigma^2}}$$
   *Box-Muller Transform:*
   $$z = \mu + \sigma \sqrt{-2 \ln U_1} \cos(2\pi U_2)$$

3. **Erlang (Gamma) Noise:**
   $$p(z) = \frac{a^b z^{b-1}}{(b-1)!} e^{-az}, \quad z \ge 0$$
   Generated as the sum of $b$ independent exponential variables:
   $$z = -\frac{1}{a} \sum_{i=1}^b \ln(U_i) = -\frac{1}{a} \ln\left(\prod_{i=1}^b U_i\right)$$

4. **Exponential Noise:**
   $$p(z) = a e^{-az}, \quad z \ge 0 \implies z = -\frac{1}{a} \ln(1 - U)$$

5. **Rayleigh Noise:**
   $$p(z) = \frac{2}{b}(z - a) e^{-\frac{(z-a)^2}{b}}, \quad z \ge a \implies z = a + \sqrt{-b \ln(1 - U)}$$


In [ ]:
# Cell 1: Import Libraries
import numpy as np
import matplotlib.pyplot as plt
import cv2

plt.rcParams['figure.figsize'] = (12, 6)


In [ ]:
# Cell 2: Test Clean Image
def get_clean_image():
    img = np.full((256, 256), 128, dtype=np.uint8)
    cv2.circle(img, (80, 80), 50, 220, -1)
    cv2.rectangle(img, (140, 140), (220, 220), 40, -1)
    cv2.putText(img, "NOISE", (60, 180), cv2.FONT_HERSHEY_SIMPLEX, 1, 180, 2)
    return img

clean_img = get_clean_image()
print("Clean reference image generated.")


In [ ]:
# Cell 3: Noise Generation from First Principles (Equations Derived from PDFs)
np.random.seed(42)

def generate_uniform_noise(shape, a=-30, b=30):
    """Derived: z = a + (b - a) * U"""
    u = np.random.uniform(0.0, 1.0, shape)
    return a + (b - a) * u

def generate_gaussian_noise(shape, mean=0, sigma=25):
    """Derived: Box-Muller transform"""
    u1 = np.random.uniform(1e-10, 1.0, shape)
    u2 = np.random.uniform(0.0, 1.0, shape)
    z0 = np.sqrt(-2.0 * np.log(u1)) * np.cos(2.0 * np.pi * u2)
    return mean + sigma * z0

def generate_erlang_noise(shape, a=0.1, b=3):
    """Derived: Sum of b exponential variables: z = -(1/a) * ln(product(U_i))"""
    product_u = np.ones(shape, dtype=np.float64)
    for _ in range(b):
        product_u *= np.random.uniform(1e-10, 1.0, shape)
    return -(1.0 / a) * np.log(product_u)

def generate_exponential_noise(shape, a=0.08):
    """Derived: z = -(1/a) * ln(1 - U)"""
    u = np.random.uniform(1e-10, 1.0, shape)
    return -(1.0 / a) * np.log(u)

def generate_rayleigh_noise(shape, a=0, b=500):
    """Derived: z = a + sqrt(-b * ln(1 - U))"""
    u = np.random.uniform(1e-10, 1.0, shape)
    return a + np.sqrt(-b * np.log(u))

print("All mathematical noise functions defined.")


In [ ]:
# Cell 4: Noise Synthesis & Image Corruption
shape = clean_img.shape

# 1. Uniform
n_uniform = generate_uniform_noise(shape, -40, 40)
img_uniform = np.clip(clean_img.astype(np.float64) + n_uniform, 0, 255).astype(np.uint8)

# 2. Gaussian
n_gaussian = generate_gaussian_noise(shape, 0, 25)
img_gaussian = np.clip(clean_img.astype(np.float64) + n_gaussian, 0, 255).astype(np.uint8)

# 3. Erlang
n_erlang = generate_erlang_noise(shape, a=0.1, b=3)
img_erlang = np.clip(clean_img.astype(np.float64) + (n_erlang - np.mean(n_erlang)), 0, 255).astype(np.uint8)

# 4. Exponential
n_exponential = generate_exponential_noise(shape, a=0.05)
img_exponential = np.clip(clean_img.astype(np.float64) + (n_exponential - np.mean(n_exponential)), 0, 255).astype(np.uint8)

# 5. Rayleigh
n_rayleigh = generate_rayleigh_noise(shape, a=0, b=600)
img_rayleigh = np.clip(clean_img.astype(np.float64) + (n_rayleigh - np.mean(n_rayleigh)), 0, 255).astype(np.uint8)

# In-built benchmark comparison (NumPy / SciPy equivalent of MATLAB imnoise)
n_gauss_inbuilt = np.random.normal(0, 25, shape)
print(f"Equation Gaussian Mean={np.mean(n_gaussian):.2f}, Std={np.std(n_gaussian):.2f}")
print(f"Inbuilt Gaussian  Mean={np.mean(n_gauss_inbuilt):.2f}, Std={np.std(n_gauss_inbuilt):.2f}")


In [ ]:
# Cell 5: Histograms & Theoretical PDF Shape Verification
noises = [
    ("Uniform Noise", n_uniform, 'royalblue'),
    ("Gaussian Noise", n_gaussian, 'forestgreen'),
    ("Erlang (Gamma) Noise", n_erlang, 'darkorange'),
    ("Exponential Noise", n_exponential, 'crimson'),
    ("Rayleigh Noise", n_rayleigh, 'purple')
]

fig, axes = plt.subplots(2, 5, figsize=(22, 9))

# Row 1: Histograms of generated noise distributions
for i, (name, n_arr, col) in enumerate(noises):
    ax = axes[0, i]
    ax.hist(n_arr.ravel(), bins=50, density=True, color=col, alpha=0.7, edgecolor='black', lw=0.5)
    ax.set_title(f"{name}\nHistogram", fontweight='bold')
    ax.set_xlabel("Noise Value")
    ax.set_ylabel("Probability Density")
    ax.grid(True, alpha=0.3)

# Row 2: Corrupted Images
noisy_imgs = [
    ("Uniform Corrupted", img_uniform),
    ("Gaussian Corrupted", img_gaussian),
    ("Erlang Corrupted", img_erlang),
    ("Exponential Corrupted", img_exponential),
    ("Rayleigh Corrupted", img_rayleigh)
]

for i, (name, c_img) in enumerate(noisy_imgs):
    ax = axes[1, i]
    ax.imshow(c_img, cmap='gray')
    ax.set_title(name, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()


### Conclusions
- The derived noise distribution equations accurately reflect the theoretical probability density functions:
  - **Uniform:** Flat probability within bounds $[a, b]$.
  - **Gaussian:** Symmetric bell curve centered at $\mu$.
  - **Erlang:** Right-skewed distribution modeling multi-stage Poisson phenomena.
  - **Exponential:** Steep exponential decay starting at zero.
  - **Rayleigh:** Asymmetric distribution with a soft peak and right-skewed tail, common in radar and MRI.
